# Round 1 (re-run) — existing checkpoint audit + two controlled fine-tuning runs

**Runtime:** GPU (T4 is enough). Use **Runtime → Run all**. Any failure raises an exception that stops the run
and names the failing stage/step and its log file. Nothing is silently skipped.

| Stage | What | Trains? | ~Time on T4 |
|---|---|---|---|
| setup | extract + verify bundle, install deps (`--use-pep517`), import check, unit tests | no | 3–5 min |
| A | `checkpoint-4352`: reproduce the original metric (≈0.9925), corrected metrics, predictions on all eval sets | no | 10–15 min |
| B | **E1**: same model/data/hyper-parameters, only the label policy fixed | yes | 25–30 min |
| C | **E2**: E1 + format-family-A augmentation | yes | 35–40 min |
| D | validate every artifact, then write `round1_results.zip` | no | 1 min |

Outputs go to `OUT` on Drive as they are produced. A stage that finished successfully is recorded in
`OUT/status/<stage>.json`. If the session disconnects, just **Run all** again: completed stages are skipped
after their status is checked, and the interrupted stage restarts from scratch.

What changed since the failed attempt: the bundle version is now `round1-r2`. See `reports/history/COLAB_ROUND1_FIX.md`.

In [ ]:
# ---- paths: edit only if your Drive layout differs ------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/confidentiality_detector"
BUNDLE = f"{DRIVE_ROOT}/colab_bundle.zip"                           # replace with the rebuilt dist/colab_bundle.zip
ORIGINAL_CKPT = f"{DRIVE_ROOT}/mbert_checkpoints/checkpoint-4352"  # read-only; its weights are hashed before/after
OUT = f"{DRIVE_ROOT}/round1_rerun"                                 # new folder; the failed 'round1' folder is not used
WORK = "/content/cd"                                               # local extracted copy of code + data
EXPECTED_BUNDLE_VERSION = "round1-r2"

## Setup 1/2 — mount Drive, check GPU, extract and verify the bundle

In [ ]:
import os, sys, shutil, zipfile
from google.colab import drive

drive.mount("/content/drive")
for label, path, kind in [("bundle", BUNDLE, "file"), ("original checkpoint", ORIGINAL_CKPT, "dir")]:
    ok = os.path.isfile(path) if kind == "file" else os.path.isdir(path)
    if not ok:
        raise FileNotFoundError(f"{label} not found at {path}")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: Runtime > Change runtime type > T4 GPU, then Run all again.")
print("GPU:", torch.cuda.get_device_name(0))

if os.path.isdir(WORK):
    shutil.rmtree(WORK)
with zipfile.ZipFile(BUNDLE) as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f"corrupt bundle member: {bad}")
    zf.extractall(WORK)

sys.path.insert(0, f"{WORK}/scripts")
import importlib
import colab_round1 as r1
importlib.reload(r1)
if r1.BUNDLE_VERSION != EXPECTED_BUNDLE_VERSION:
    raise RuntimeError(f"bundle version {r1.BUNDLE_VERSION} != {EXPECTED_BUNDLE_VERSION}: upload the rebuilt colab_bundle.zip")
ctx = r1.Context(work=WORK, out=OUT, original_ckpt=ORIGINAL_CKPT)
os.makedirs(OUT, exist_ok=True)
r1.verify_bundle(ctx)

## Setup 2/2 — dependencies, import check in a fresh interpreter, unit tests
`install_dependencies` runs `python -m pip install --use-pep517 -r requirements-colab.txt` and fails on a non-zero exit.
`check_environment` imports every required package **and** project module in a new process and writes `OUT/environment.json`.
`run_tests` runs the whole test suite; any failure stops here, before GPU time is spent.

In [ ]:
r1.install_dependencies(ctx)
env_info = r1.check_environment(ctx, require_gpu=True)
r1.run_tests(ctx)
print("setup OK:", {k: env_info["versions"].get(k) for k in ("torch", "transformers", "seqeval", "phonenumbers", "pip")})

## Stage A — existing `checkpoint-4352` (no training)
Prints four rows on the validation split: **A** = original notebook metric (expected ≈ 0.9925 F1),
**B** = ignored positions dropped, **C** = document-level spans with the original gold, **D** = corrected gold.
It then predicts on all 9 evaluation sets and scores them. The checkpoint's weight files are hashed before and after.

In [ ]:
r1.run_stage(ctx, "A")

## Stage B — E1: v2 label policy only (`configs/experiments/e1_mbert_v2.json`)
Each epoch is checkpointed on local disk and scored on validation. The best epoch is **reloaded from disk and must reproduce its
validation F1**, or the stage fails. Only then is it copied to `OUT/checkpoints/e1_mbert_v2`.

In [ ]:
r1.run_stage(ctx, "B")

## Stage C — E2: E1 + format-family-A augmentation (`configs/experiments/e2_mbert_v2_fmtA.json`)

In [ ]:
r1.run_stage(ctx, "C")

## Stage D — package results (predictions, metrics, logs, status, training histories; **no weights**)
Refuses to run unless A, B and C are all `ok` **and** their artifacts re-validate. The archive is re-opened and checked
after writing. Download `OUT/round1_results.zip`.

In [ ]:
archive = r1.package(ctx)
print("Send this file back:", archive)